In [1]:
!pip install -q FlagEmbedding

import pandas as pd
import numpy as np
from FlagEmbedding import BGEM3FlagModel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 67.2 MB/s eta 0:00:00


In [2]:
BASE_PATH = '/content/drive/MyDrive/Sanjana/Maintenance_Log_Intelligence/ASRS/processed'
df = pd.read_csv(f'{BASE_PATH}/asrs_cleaned.csv')

print(f"Loaded {len(df)} reports")
df.head(2)

Loaded 9443 reports


,acn,narrative_combined,synopsis,callback_combined,narrative_word_count,aircraft_model,flight_phase,mission,aircraft_operator,airport,state,anomaly_events,report_year,report_date
0,818235.0,Upon reaching 16;000 FT MSL and completion of ...,Q400 flight crew experiences partial opening o...,NaN,195,Dash 8-400,Cruise,Passenger,Air Carrier,ZZZ.Airport,US,Aircraft Equipment Problem Less Severe,2009,200901
1,818294.0,During cruise at FL340; the Captain and I were...,An ERJ170 flight crew got EICAS message FUEL F...,NaN,549,EMB ERJ 170/175 ER/LR,Cruise,Passenger,Air Carrier,ZZZ.Airport,US,Aircraft Equipment Problem Critical,2009,200901


In [3]:
# Build the chunk records (one narrative chunk + one synopsis chunk per report)

chunks = []

for _, row in df.iterrows():
    chunks.append({
        'chunk_id': f"{row['acn']}_narrative",
        'acn': row['acn'],
        'text': row['narrative_combined'],
        'chunk_type': 'narrative',
        'aircraft_model': row['aircraft_model'],
        'flight_phase': row['flight_phase'],
        'mission': row['mission'],
        'aircraft_operator': row['aircraft_operator'],
        'airport': row['airport'],
        'state': row['state'],
        'report_year': row['report_year']
    })

    if pd.notna(row['synopsis']) and len(str(row['synopsis']).split()) > 5:
        chunks.append({
            'chunk_id': f"{row['acn']}_synopsis",
            'acn': row['acn'],
            'text': row['synopsis'],
            'chunk_type': 'synopsis',
            'aircraft_model': row['aircraft_model'],
            'flight_phase': row['flight_phase'],
            'mission': row['mission'],
            'aircraft_operator': row['aircraft_operator'],
            'airport': row['airport'],
            'state': row['state'],
            'report_year': row['report_year']
        })

chunks_df = pd.DataFrame(chunks)
print(f"Total chunks created: {len(chunks_df)}")
print(chunks_df['chunk_type'].value_counts())

Total chunks created: 18886
chunk_type
narrative    9443
synopsis     9443
Name: count, dtype: int64


In [4]:
# Load BGE-M3 model

model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
print("Model loaded")

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model loaded


In [5]:
# Generate embeddings in batches

texts = chunks_df['text'].astype(str).tolist()

output = model.encode(
    texts,
    batch_size=32,
    max_length=2048,
    return_dense=True,
    return_sparse=False,
    return_colbert_vecs=False
)

embeddings = output['dense_vecs']
print(f"Embeddings shape: {embeddings.shape}")

Inference Embeddings: 100%|██████████| 591/591 [05:11<00:00,  1.90it/s]


Embeddings shape: (18886, 1024)


In [6]:
# Save embeddings and metadata to Drive

np.save(f'{BASE_PATH}/asrs_embeddings.npy', embeddings)
chunks_df.to_csv(f'{BASE_PATH}/asrs_chunks_metadata.csv', index=False)

print(f"Saved {embeddings.shape[0]} embeddings and metadata to {BASE_PATH}")

Saved 18886 embeddings and metadata to /content/drive/MyDrive/Sanjana/Maintenance_Log_Intelligence/ASRS/processed
